Este notebook irá selecionar na biblioteca geobr as informações de Latitute e Longitude dos Municípios

Separar as informações geoespaciais - Latitude, Logitude e Coordenadas

In [ ]:
import geobr
# import geopandas as gpd
import os, sys
# from pyspark.sql import functions as F

In [ ]:
# Adiciona a pasta raiz do projeto (um ou dois níveis acima) no caminho do Python
sys.path.append(os.path.abspath(os.path.join('..')))  # Ajuste a quantidade de '..' conforme a profundidade da subpasta

# Cria uma conexão Spark 
from spark_utils import get_spark_session # ver em C:\Marco Conti\Projetos\MAIS-v2\spark_utils.py
spark = get_spark_session("MeuNotebook")


In [ ]:
# Municípios
mun = geobr.read_municipality(year=2024)

# Reprojetar para um CRS métrico
mun_proj = mun.to_crs("EPSG:5880")

# Gerar ponto representativo
mun_proj["rep_point"] = mun_proj.geometry.representative_point()

# Voltar para latitude/longitude
mun["rep_point"] = mun_proj["rep_point"].to_crs("EPSG:4674")


In [ ]:
# Formatar colunas
mun["longitude"]    = (mun["rep_point"].x).astype(float)
mun["latitude"]     = (mun["rep_point"].y).astype(float)
mun["geometry"]     = mun["geometry"].astype(str)
mun["code_muni"]    = mun["code_muni"].astype(int)
mun["year"]         = mun["year"].astype(int)


In [ ]:
# Selecionar apenas as colunas desejadas
df = mun[
    [
        "code_muni",
        "name_muni",
        "abbrev_state",
        "name_state",
        "name_region",
        "year",
        "latitude",
        "longitude",
        "geometry"
    ]
]

In [ ]:
df = \
    (df.rename(columns={'code_muni'     : 'codigo_municipio'
                       ,'name_muni'     : 'nome_municipio'
                       ,'abbrev_state'  : 'UF'
                       ,'name_state'    : 'nome_estado'
                       ,'name_region'   : 'grande_regiao'
                       ,'year'          : 'ano'
                       ,'latitude'      : 'latitude'
                       ,'longitude'     : 'longitude'
                       ,'geometry'      : 'coordenadas_geoespaciais'}))

In [ ]:
# Converter o dataframe para Spark

df_coord_geo = spark.createDataFrame(df)
df_coord_geo.printSchema()
df_coord_geo.show(10,False)

In [ ]:
# Salvar em CSV
df.to_csv(r"C:\\Marco Conti\\Projetos\\MAIS-v2\\dados\\municipios\\municipios_coord_geograficas.csv"
         ,index=False
         ,encoding="utf-8-sig")

#Salvar em Parquet
df.to_parquet(r"C:\\Marco Conti\\Projetos\\MAIS-v2\\dados\\municipios\\municipios_coord_geograficas.parquet")

In [ ]:
# df_x = spark.read.csv(r"C:\\Marco Conti\\Projetos\\MAIS-v2\\dados\\municipios\\municipios_coord_geograficas.csv", header=True, inferSchema=True)
# df_x.printSchema()
# df_x.show(10,False)

df_y = spark.read.parquet(r"C:\Marco Conti\Projetos\MAIS-v2\dados\municipios\municipios_coord_geograficas.parquet")
df_y.printSchema()
df_y.show(10,False)
